In [ ]:
# Required Libraries
import os
import asyncio
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ Environment loaded")

## 1. Azure OpenAI Configuration

In [ ]:
# Azure OpenAI credentials from environment
AZURE_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT", "https://your-endpoint.cognitiveservices.azure.com/")
DEPLOYMENT_NAME = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME", "gpt-4.1")
API_VERSION = os.getenv("AI_FOUNDRY_API_VERSION", "2024-12-01-preview")
API_KEY = os.getenv("AI_FOUNDRY_API_KEY", "")

print("🔧 Azure OpenAI Configuration:")
print(f"   Endpoint: {AZURE_ENDPOINT}")
print(f"   Deployment: {DEPLOYMENT_NAME}")
print(f"   API Version: {API_VERSION}")
print(f"   API Key: {'***' + API_KEY[-4:] if API_KEY else 'NOT SET'}")

if not API_KEY:
    print("\n⚠️ WARNING: API_KEY not set. Please set AI_FOUNDRY_API_KEY in .env file")

## 2. Create Azure OpenAI Client

In [ ]:
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

def create_model_client():
    """
    Create Azure OpenAI model client for Autogen.
    Uses the new Autogen 0.4+ API.
    """
    client = AzureOpenAIChatCompletionClient(
        azure_endpoint=AZURE_ENDPOINT,
        model=DEPLOYMENT_NAME,
        api_version=API_VERSION,
        api_key=API_KEY,
    )
    return client

# Create client
model_client = create_model_client()
print("✅ Azure OpenAI client created")

## 3. Create a Single Agent

In [ ]:
from autogen_agentchat.agents import AssistantAgent

# Define a Data Scientist agent
data_scientist = AssistantAgent(
    name="DataScientist",
    model_client=model_client,
    system_message="""
    You are a senior data scientist with expertise in:
    - Exploratory Data Analysis (EDA)
    - Statistical analysis
    - Machine learning
    - Data visualization
    
    When given a dataset description, provide:
    1. Key observations
    2. Recommended analyses
    3. Potential issues to investigate
    
    Be concise and practical.
    """
)

print(f"✅ Agent created: {data_scientist.name}")

## 4. Run Agent (Single Response)

In [ ]:
from autogen_core import CancellationToken

async def run_agent_single(agent, message: str):
    """
    Run an agent with a single message and get response.
    """
    print(f"\n📨 User: {message}")
    print("\n" + "="*50)
    
    # Create cancellation token
    token = CancellationToken()
    
    # Run agent
    response = await agent.run(
        task=message,
        cancellation_token=token
    )
    
    # Get the last message from the response
    if response.messages:
        last_msg = response.messages[-1]
        print(f"\n🤖 {agent.name}:")
        print(last_msg.content)
    
    return response

In [ ]:
# Test the agent
test_message = """
I have a Titanic dataset with the following columns:
- PassengerId, Survived, Pclass, Name, Sex, Age, SibSp, Parch, Ticket, Fare, Cabin, Embarked

The dataset has 891 rows. Age has some missing values.
My goal is to predict survival.

What should I analyze first?
"""

# Run the agent
response = await run_agent_single(data_scientist, test_message)

## 5. Create Multiple Specialized Agents

In [ ]:
# Create specialized agents

eda_agent = AssistantAgent(
    name="EDAExpert",
    model_client=model_client,
    system_message="""
    You are an EDA (Exploratory Data Analysis) expert.
    Focus on:
    - Data quality assessment
    - Distribution analysis
    - Correlation analysis
    - Missing value patterns
    
    Provide specific, actionable insights.
    """
)

ml_agent = AssistantAgent(
    name="MLEngineer",
    model_client=model_client,
    system_message="""
    You are a machine learning engineer.
    Focus on:
    - Model selection
    - Feature engineering
    - Training pipelines
    - Model evaluation
    
    Recommend appropriate algorithms and techniques.
    """
)

viz_agent = AssistantAgent(
    name="DataVisualizer",
    model_client=model_client,
    system_message="""
    You are a data visualization expert.
    Focus on:
    - Chart selection
    - Visual storytelling
    - Dashboard design
    - Matplotlib/Seaborn best practices
    
    Recommend specific visualizations for the data.
    """
)

print("✅ Specialized agents created:")
print(f"   - {eda_agent.name}")
print(f"   - {ml_agent.name}")
print(f"   - {viz_agent.name}")

In [ ]:
# Test each agent with the same question
question = "How should I handle the missing Age values in Titanic dataset for survival prediction?"

print("="*60)
print("🔄 Getting responses from all agents...")
print("="*60)

In [ ]:
# EDA Expert response
await run_agent_single(eda_agent, question)

In [ ]:
# ML Engineer response
await run_agent_single(ml_agent, question)

In [ ]:
# Data Visualizer response
await run_agent_single(viz_agent, question)

## 6. Agent with Code Generation

In [ ]:
# Create a coding-focused agent
coder_agent = AssistantAgent(
    name="PythonCoder",
    model_client=model_client,
    system_message="""
    You are a Python coding expert for data science.
    
    When asked to write code:
    1. Use pandas, numpy, sklearn, matplotlib, seaborn
    2. Include comments explaining each step
    3. Handle edge cases
    4. Follow PEP 8 style
    
    Provide working, runnable code.
    """
)

print(f"✅ Created: {coder_agent.name}")

In [ ]:
# Ask for code
code_request = """
Write Python code to:
1. Load the Titanic dataset
2. Handle missing Age values using median imputation
3. Create a simple survival prediction model using Random Forest
4. Print the accuracy score
"""

await run_agent_single(coder_agent, code_request)

## ✅ Summary

This module demonstrates:

**1. Azure OpenAI Setup**
- Environment variable configuration
- `AzureOpenAIChatCompletionClient` from autogen_ext

**2. Single Agent Creation**
- `AssistantAgent` with custom system message
- Running agent with `agent.run()`
- Using `CancellationToken` for async control

**3. Specialized Agents**
- Different agents for different tasks
- EDA, ML, Visualization, Coding

**4. Key Autogen 0.4+ Imports**
```python
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient
from autogen_core import CancellationToken
```